usage see peters/GlitchTimingExample-interactive-timing-example.ipynb

In [ ]:
%matplotlib widget
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
import time

def plot_la_interactive(t, sigs, names, unit="ns", snap=True, title=""):
    scale = {"s": 1.0, "ms": 1e-3, "us": 1e-6, "ns": 1e-9}[unit]   # unit -> seconds
    n_ch = len(sigs)
    fig, axes = plt.subplots(n_ch, 1, figsize=(11, 1 + 0.7 * n_ch), sharex=True, squeeze=False)
    axs = axes[:, 0]

    edge_times = []
    for ax, name, sig in zip(axs, names, sigs):
        sig = np.asarray(sig)
        ax.step(t, sig, where="post")
        ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
        ax.set_yticks([0, 1])
        ax.set_ylim(-0.2, 1.2)
        ax.grid(True, alpha=0.3)
        edge_times.append(t[np.flatnonzero(np.diff(sig) != 0) + 1])
    edges = np.sort(np.concatenate(edge_times))

    axs[-1].set_xlabel(f"time ({unit})")
    help_txt = "Left-click: cursor A/B | Right-click: clear"
    fig.suptitle(f"{title}\n{help_txt}" if title else help_txt, fontsize=9)

    marks, lines = [], []

    def clear():
        for ln in lines:
            ln.remove()
        lines.clear()
        marks.clear()

    def on_click(event):
        tb = fig.canvas.toolbar
        if tb is not None and getattr(tb, "mode", "") != "":
            return                      # zoom/pan tool active
        if event.inaxes not in axs:
            return
        if event.button == 3:
            clear()
            fig.suptitle(f"{title}\n{help_txt}" if title else help_txt, fontsize=9)
            fig.canvas.draw_idle()
            return
        if event.button != 1:
            return

        x = event.xdata
        if snap and edges.size:
            x = edges[np.argmin(np.abs(edges - x))]
        if len(marks) == 2:
            clear()
        marks.append(x)
        color = "tab:red" if len(marks) == 1 else "tab:green"
        for ax in axs:
            lines.append(ax.axvline(x, color=color, linestyle="--", lw=1))

        if len(marks) == 1:
            msg = f"A = {x:.2f} {unit}"
        else:
            dt = abs(marks[1] - marks[0])
            f = 1 / (dt * scale) if dt > 0 else float("nan")
            msg = (f"A = {marks[0]:.2f} {unit}   B = {marks[1]:.2f} {unit}   "
                   f"Δt = {dt:.2f} {unit}   1/Δt = {f/1e6:.3f} MHz")
        fig.suptitle(msg, fontsize=10)
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.tight_layout()
    plt.show()
    return fig


def pick_time_unit(span_s):
    """Return (unit_name, factor) so that span_s * factor is a readable number."""
    for name, factor in (("s", 1.0), ("ms", 1e3), ("us", 1e6), ("ns", 1e9)):
        if span_s * factor >= 1:        # span is at least 1 of this unit
            return name, factor
    return "ps", 1e12

def calc_la_time_axis_in_s(scope, sigs):
    # the time axis in seconds, from the real LA sample rate
    f_la = scope.LA.sampling_clock_frequency / scope.LA.downsample
    n = len(sigs[0])
    t_s = np.arange(n) / f_la
    return t_s, n

def calc_la_unit(scope, sigs):
    t_s, n = calc_la_time_axis_in_s(scope, sigs)
    unit, factor = pick_time_unit(t_s[-1])   # based on the total capture length
    t = t_s * factor
    print(f"{n} samples, span = {t[-1]:.2f} {unit}, sample period = {1/f_la*factor:.2f} {unit}")
    return unit

LA_NAMES = {
    "glitch": [
        "glitch output",
        "glitch src clock",
        "MMCM1 (offset) out",
        "MMCM2 (width) out",
        "glitch go",
        "capture trigger",
        "glitch enable",
        "manual trig (src clk)",
        "trig in MMCM1 clk",
    ],
    "CW 20-pin": ["IO1", "IO2", "IO3", "IO4", "HS1", "HS2",
                  "AUX MCX", "TRIG MCX", "ADC sampling clock"],
    "USERIO 20-pin": ["D0", "D1", "D2", "D3", "D4", "D5", "D6", "D7", "CK"],
}


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt


IDX = {n: i for i, n in enumerate(LA_NAMES["glitch"])}


def edges(sig, kind="rise"):
    """Sample indices of rising or falling edges."""
    d = np.diff(np.asarray(sig).astype(int))
    return np.flatnonzero(d == (1 if kind == "rise" else -1)) + 1


def glitch_times(ext_offset, offset, width, repeat, steps, f_glitch, output="enable_only", latency_ns=0.0):
    """Convert glitch settings to times in ns (computed, not measured).

    latency_ns: fixed trigger->output latency you measured (about 430 ns for you).
    """
    T = 1 / f_glitch
    ns = T * 1e9
    d = dict(
        T_ns=ns,
        ext_offset_ns=ext_offset * ns,            # whole clock cycles after the trigger
        offset_ns=offset / steps * ns,            # fine phase shift of the glitch position
        width_ns=width / steps * ns,              # fine phase shift: pulse width (non-enable_only modes)
        enable_ns=repeat * ns,                    # length of the enable window, repeat cycles
    )
    d["start_ns"] = d["ext_offset_ns"] + d["offset_ns"] + latency_ns
    if output == "enable_only":
        d["pulse_ns"] = d["enable_ns"]            # width/offset do not change this output
    else:
        d["pulse_ns"] = d["width_ns"]             # per glitch cycle, if my reading of the mode is right
    return d

def fmt_ns(x):
    if x != x:                      # NaN
        return "n/a"
    if x >= 100:
        return f"{x:.0f} ns"
    if x >= 1:
        return f"{x:.1f} ns"
    return f"{x * 1000:.0f} ps"
    
def caption(res):
    """Text with the computed times and, if found, the measured ones."""
    tm = res["times"]
    s = (f"ext_offset={res['ext_offset']} ({fmt_ns(tm['ext_offset_ns'])})  "
     f"offset={res['offset']} ({fmt_ns(tm['offset_ns'])})  "
     f"width={res['width']} ({fmt_ns(tm['width_ns'])})  "
     f"repeat={res['repeat']} (pulse {fmt_ns(tm.get('pulse_ns', float('nan')))})")
    s += f"\ncomputed start={tm['start_ns']:.0f} ns"
    if "delay_ns" in res:
        s += f"   measured: delay={res['delay_ns']:.0f} ns"
        if "pulse_ns" in res:
            s += f"  pulse={res['pulse_ns']:.0f} ns"
    return s

def pin_set(setting):
    scope.io.tio4 = setting

def pre_arm_action(scope):
    pin_set("gpio_low")

def post_arm_trigger_action(scope):
    pin_set("gpio_high")                   # rising edge on the trigger pin

def post_capture_action(scope):
    pin_set("gpio_low")


def run_glitch(scope, ext_offset=1, repeat=1, offset=0, width=1000,
               output="enable_only", oversampling=10, extra_us=2.0,
               trig_pin="tio4", f_glitch=10e6, fixed_latency_us=0.6):
    """Configure the glitch module, fire one glitch, capture it with the LA.

    f_glitch         : glitch clock in Hz (must match clkgen_freq / PLL setup)
    fixed_latency_us : margin for the fixed trigger->output latency
                       (about 0.43 us measured on your setup, plus some slack)
    Returns a dict with the LA data, computed times and measured timing.
    """
    # ---- glitch module ----
    g = scope.glitch
    g.enabled = True
    g.clk_src = "pll"
    g.trigger_src = "ext_single"
    g.num_glitches = 1
    g.output = output
    g.ext_offset = ext_offset
    g.repeat = repeat
    g.offset = offset
    g.width = width

    # ---- logic analyzer ----
    la = scope.LA
    la.enabled = True
    la.clk_source = "pll"
    la.capture_group = "glitch"
    la.trigger_source = "trigger_glitch"   # the event ext_offset is counted from
    la.oversampling_factor = oversampling
    la.downsample = 1

    f_la = la.sampling_clock_frequency / la.downsample
    T = 1 / f_glitch
    window = (ext_offset + repeat) * T + (extra_us + fixed_latency_us) * 1e-6
    la.capture_depth = int(window * f_la)

    # ---- fire one glitch ----
    pin_set = lambda v: setattr(scope.io, trig_pin, v)
    pre_arm_action(scope)
    time.sleep(0.01)
    la.arm()
    scope.arm()

    post_arm_trigger_action(scope)
    raw = la.read_capture_data(check_empty=False)
    post_capture_action(scope)

    sigs = [la.extract(raw, i) for i in range(len(LA_NAMES["glitch"]))]

    # ---- measure glitch output ----
    out = sigs[IDX["glitch output"]]
    r, f = edges(out, "rise"), edges(out, "fall")

    res = dict(
        ext_offset=ext_offset, repeat=repeat, offset=offset, width=width,
        n_pulses=len(r), f_la=f_la, sigs=sigs,
        t=np.arange(len(out)) / f_la,
        times=glitch_times(ext_offset, offset, width, repeat,
                   g.phase_shift_steps, f_glitch, output, latency_ns=430),
    )
    if len(r):
        res["delay_ns"] = r[0] / f_la * 1e9
        later_falls = f[f > r[0]]
        if len(later_falls):
            res["pulse_ns"] = (later_falls[0] - r[0]) / f_la * 1e9
    return res


def sweep(scope, param, values, **fixed):
    """Vary one parameter, print a table, overlay the glitch output traces."""
    results = []
    for v in values:
        res = run_glitch(scope, **{**fixed, param: v})
        results.append(res)
        print(f"{param}={v:<6} pulses={res['n_pulses']}  "
              f"delay={res.get('delay_ns', float('nan')):8.0f} ns  "
              f"pulse={res.get('pulse_ns', float('nan')):8.0f} ns")

    fig, ax = plt.subplots(figsize=(11, 1.5 + 0.6 * len(values)))
    for k, res in enumerate(results):
        out = np.asarray(res["sigs"][IDX["glitch output"]])
        ax.step(res["t"] * 1e9, out * 0.8 + k, where="post")
    ax.set_yticks([k + 0.4 for k in range(len(values))])
    ax.set_yticklabels([f"{param}={v}" for v in values])
    ax.set_xlabel("time since glitch trigger (ns)")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    return results